In [1]:
from datasets import load_dataset
import pandas as pd
import re
# Reload dataset
dataset = load_dataset("tasksource/logical-fallacy")

# Convert to dataframes
df_train = pd.DataFrame(dataset['train']).rename(columns={'source_article': 'text', 'logical_fallacies': 'label'})
df_test = pd.DataFrame(dataset['test']).rename(columns={'source_article': 'text', 'logical_fallacies': 'label'})
df_dev = pd.DataFrame(dataset['dev']).rename(columns={'source_article': 'text', 'logical_fallacies': 'label'})

print("✅ Data loaded successfully")
print(f"Train: {len(df_train)} | Test: {len(df_test)} | Dev: {len(df_dev)}")


C:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Data loaded successfully
Train: 2680 | Test: 511 | Dev: 570


In [2]:
print("=== Null Values ===")
print(df_train.isnull().sum())

=== Null Values ===
config    0
text      0
label     0
dtype: int64


In [3]:
print("\n=== Duplicates ===")
print(f"Train: {df_train.duplicated(subset='text').sum()}")
print(f"Test : {df_test.duplicated(subset='text').sum()}")
print(f"Dev  : {df_dev.duplicated(subset='text').sum()}")


=== Duplicates ===
Train: 163
Test : 33
Dev  : 53


In [4]:
print("\n=== Very Short Texts (less than 5 words) ===")
short = df_train[df_train['text'].apply(lambda x: len(x.split()) < 5)]
print(f"Count: {len(short)}")
print(short[['text', 'label']].head(10))


=== Very Short Texts (less than 5 words) ===
Count: 6
                           text                   label
551   Is your stupidity inborn?             intentional
579        Two words: New Coke.  fallacy of credibility
1177  Photo shows flying saucer        fallacy of logic
1763              men don’t cry   faulty generalization
2332              Big trouble .       appeal to emotion
2601            But will they ?             intentional


In [5]:
print("\n=== Very Short Texts (<5 words) ===")
short = df_train[df_train['text'].apply(lambda x: len(str(x).split()) < 5)]
print(f"Count: {len(short)}")
print(short[['text', 'label']].head())


=== Very Short Texts (<5 words) ===
Count: 6
                           text                   label
551   Is your stupidity inborn?             intentional
579        Two words: New Coke.  fallacy of credibility
1177  Photo shows flying saucer        fallacy of logic
1763              men don’t cry   faulty generalization
2332              Big trouble .       appeal to emotion


In [6]:
print("\n=== Very Long Texts (>100 words) ===")
long = df_train[df_train['text'].apply(lambda x: len(str(x).split()) > 100)]
print(f"Count: {len(long)}")


=== Very Long Texts (>100 words) ===
Count: 29


In [7]:
print("\n=== Random Samples ===")
for t in df_train['text'].sample(5).tolist():
    print("-", t)


=== Random Samples ===
- Same-sex marriage must be prohibited, or the family structure as we know it will  collapse.
- The picture on Jim's old TV set goes out of focus. Jim goes over and strikes the TV soundly on the side and the picture goes back into focus. Jim tells his friend that hitting the TV fixed it.
- The slowdown in warming was , she added , real , and all the evidence suggested that since 1998 , the rate of global warming has been much slower than predicted by computer models – about 1C per century .
- Although we have proven that the moon is not made of short ribs, we have not proven that its core cannot be filled with them; therefore, the moon’s core is filled with short ribs.
- Either we go to war or we appear weak.


In [8]:
#data cleaning
def clean_text(text):
    text=str(text)
    text=text.replace('\n',' ')
    text=re.sub(r'["""]','""',text)
    text=re.sub(r"['']","'",text)
    text=re.sub(r'\s+',' ',text)
    text=text.strip()
    return text

def clean_df(df,name):
    original_size=len(df)
    df=df.copy()
    df['text']=df['text'].apply(clean_text)

    df=df.drop_duplicates(subset='text').reset_index(drop=True)

    df=df[df['text'].apply(lambda x:len(x.split())>=5)].reset_index(drop=True)

    print(f"{name}:{original_size}->{len(df)} rows (removed{original_size-len(df)})")
    return df
    
df_train=clean_df(df_train,"train")
df_test=clean_df(df_test,"test")
df_dev=clean_df(df_dev,"dev")
      

train:2680->2492 rows (removed188)
test:511->477 rows (removed34)
dev:570->515 rows (removed55)


In [9]:
print("=== After Cleaning ===")
print(f"Train duplicates remaining: {df_train.duplicated(subset='text').sum()}")
print(f"Test duplicates remaining : {df_test.duplicated(subset='text').sum()}")
print(f"Dev duplicates remaining  : {df_dev.duplicated(subset='text').sum()}")

print("\n=== Final Label Distribution (Train) ===")
print(df_train['label'].value_counts())

# Save cleaned data to CSV
df_train.to_csv('train_clean.csv', index=False)
df_test.to_csv('test_clean.csv', index=False)
df_dev.to_csv('dev_clean.csv', index=False)

print("\n Cleaned data saved to CSV files!")

=== After Cleaning ===
Train duplicates remaining: 0
Test duplicates remaining : 0
Dev duplicates remaining  : 0

=== Final Label Distribution (Train) ===
label
faulty generalization     379
intentional               278
ad hominem                272
ad populum                206
appeal to emotion         204
false causality           204
fallacy of credibility    182
fallacy of relevance      159
fallacy of logic          157
circular reasoning        138
false dilemma             138
fallacy of extension      124
equivocation               51
Name: count, dtype: int64

 Cleaned data saved to CSV files!
